# Minio - DuckDB Connection

In [1]:
import duckdb

# 1. Connect to your database
con = duckdb.connect(r"C:\data\my_warehouse.duckdb", read_only=True) 
# Note: If you get an error installing extensions on a read_only database, 
# remove `read_only=True` temporarily.

# 2. Install extensions and configure MinIO connection
con.execute("""
    INSTALL httpfs;
    LOAD httpfs;
    INSTALL iceberg;
    LOAD iceberg;
    
    SET s3_endpoint='127.0.0.1:9000';
    SET s3_access_key_id='minioadmin';
    SET s3_secret_access_key='minioadmin';
    SET s3_url_style='path';
    SET s3_use_ssl=false;
""")

# Select

In [2]:
TestSQL = """
SELECT 
    city.WWI_City_ID AS "City Code",
    city.City AS "City Name",
    city.State_Province AS "State/Province",
    city.Country AS "Country",
    city.Continent AS "Continent",
    city.Sales_Territory AS "Sales Territory",
    city.Region AS "Region",
    city.Subregion AS "Subregion",
    city.Latest_Recorded_Population AS "Latest Recorded Population",
    customer.WWI_Customer_ID AS "Customer Code",
    customer.Customer AS "Customer Name",
    customer.Bill_To_Customer AS "Billing Customer Name",
    customer.Category AS "Customer Category",
    customer.Buying_Group AS "Buying Customer Group",
    customer.Primary_Contact AS "Primary Contact",
    customer.Postal_Code AS "Postal Code",
    stockitem.WWI_Stock_Item_ID AS "Stock Item Code",
    stockitem.Stock_Item AS "Stock Item Name",
    stockitem.Color AS "Item Color",
    stockitem.Selling_Package AS "Item Selling Package",
    stockitem.Buying_Package AS "Item Buying Package",
    stockitem.Brand AS "ItemBrand",
    stockitem.Size As "Item Size",
    stockitem.Lead_Time_Days AS "Item Lead Time (Days)",
    stockitem.Quantity_Per_Outer AS "Item Quantity Per Outer",
    stockitem.Is_Chiller_Stock AS "Is Chiller Item Stock",
    stockitem.Barcode AS "Item Barcode",
    stockitem.Tax_Rate AS "Item Tax Rate",
    stockitem.Unit_Price AS "Item Unit Price",
    stockitem.Recommended_Retail_Price AS "Item Recommended Retail Price",
    stockitem.Typical_Weight_Per_Unit AS "Item Typical Weight Per Unit",
    SalesPerson.WWI_Employee_ID AS "Sales Person Code",
    SalesPerson.Employee AS "Sales Person Name",
    SalesPerson.Preferred_Name AS "Sales Person Preferred Name",
    SalesPerson.Is_Salesperson AS "Is Sales Person",
    PickerPerson.WWI_Employee_ID AS "Picker Person Code",
    PickerPerson.Employee AS "Picker Person Name",
    PickerPerson.Preferred_Name AS "Picker Person Preferred Name",
    PickerPerson.Is_Salesperson AS "Is Picker Person",
    orders.Order_Date_Key,
    orders.Picked_Date_Key,
    orders.WWI_Order_ID AS "Order Code",
    orders.WWI_Backorder_ID AS "Order Backorder Code",
    orders.Description AS "Order Description",
    orders.Package AS "Order Package",
    orders.Quantity AS "Order Quantity",
    orders.Unit_Price AS "Order Unit Price",
    orders.Tax_Rate AS "Order Tax Rate",
    orders.Total_Excluding_Tax AS "Order Total Excluding Tax",
    orders.Tax_Amount AS "Order Tax Amount",
    orders.Total_Including_Tax AS "Order Total Including Tax"
FROM 
    iceberg_scan('s3://iceberg/iceberg/WideWorldImportersDW/fact/order', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) AS orders LEFT OUTER JOIN
    iceberg_scan('s3://iceberg/iceberg/WideWorldImportersDW/Dimension/City', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS city ON orders.City_Key = city.City_Key LEFT OUTER JOIN
    iceberg_scan('s3://iceberg/iceberg/WideWorldImportersDW/Dimension/Customer', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS customer ON orders.Customer_Key = customer.Customer_Key LEFT OUTER JOIN
    iceberg_scan('s3://iceberg/iceberg/WideWorldImportersDW/Dimension/Stock_Item', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS stockitem ON orders.Stock_Item_Key = stockitem.Stock_Item_Key LEFT OUTER JOIN
    iceberg_scan('s3://iceberg/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS SalesPerson ON orders.Salesperson_Key = SalesPerson.Employee_Key LEFT OUTER JOIN
    iceberg_scan('s3://iceberg/iceberg/WideWorldImportersDW/Dimension/Employee', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN)))  AS PickerPerson ON orders.Salesperson_Key = PickerPerson.Employee_Key AND orders.Picker_Key = PickerPerson.Employee_Key
LIMIT 10
"""

# TestSQL = """
# Select * from iceberg_scan('s3://iceberg/iceberg/WideWorldImportersDW/fact/order', ("version" = '?'), (allow_moved_paths = CAST('t' AS BOOLEAN))) LIMIT 10
# """

# 4. Execute the query
df_objects = con.execute(TestSQL).df()

# 5. Display and close
display(df_objects)


,City Code,City Name,State/Province,Country,Continent,Sales Territory,Region,Subregion,Latest Recorded Population,Customer Code,...,Order Code,Order Backorder Code,Order Description,Order Package,Order Quantity,Order Unit Price,Order Tax Rate,Order Total Excluding Tax,Order Tax Amount,Order Total Including Tax
0,7890,Cramerton,North Carolina,United States,North America,Southeast,Americas,Northern America,4165,0,...,32336,32373,Bubblewrap dispenser (Blue) 1.5m,Each,1,240.0,15.0,240.0,36.00,276.00
1,31323,Shell Knob,Missouri,United States,North America,Plains,Americas,Northern America,1379,0,...,49873,49928,"""The Gu"" red shirt XML tag t-shirt (Black) 7XL",Each,72,18.0,15.0,1296.0,194.40,1490.40
2,20047,Lostine,Oregon,United States,North America,Far West,Americas,Northern America,213,563,...,3162,3242,Shipping carton (Brown) 457x457x457mm,Each,50,2.1,15.0,105.0,15.75,120.75
3,29490,Rosa Sánchez,Puerto Rico (US Territory),United States,North America,External,Americas,Northern America,1055,431,...,37595,37606,Superhero action jacket (Blue) L,Each,4,30.0,15.0,120.0,18.00,138.00
4,15248,Herlong,California,United States,North America,Far West,Americas,Northern America,298,420,...,65830,65855,USB food flash drive - banana,Each,8,3.2,15.0,25.6,3.84,29.44
5,34584,Tunnelhill,Pennsylvania,United States,North America,Mideast,Americas,Northern America,363,100,...,8187,8259,Plush shark slippers (Gray) M,Each,6,32.0,15.0,192.0,28.80,220.80
6,17353,Karthaus,Pennsylvania,United States,North America,Mideast,Americas,Northern America,0,476,...,28312,28351,DBA joke mug - it depends (Black),Each,5,13.0,15.0,65.0,9.75,74.75
7,4291,Brown City,Michigan,United States,North America,Great Lakes,Americas,Northern America,1325,187,...,60950,60967,USB food flash drive - pizza slice,Each,8,32.0,15.0,256.0,38.40,294.40
8,25078,Oakpark,Virginia,United States,North America,Southeast,Americas,Northern America,0,0,...,39859,39940,"""The Gu"" red shirt XML tag t-shirt (Black) L",Each,84,18.0,15.0,1512.0,226.80,1738.80
9,36513,West Frostproof,Florida,United States,North America,Southeast,Americas,Northern America,0,569,...,46011,46051,Superhero action jacket (Blue) S,Each,3,25.0,15.0,75.0,11.25,86.25


In [3]:
con.close()